# Create Fondazione Umberto Veronesi Awards

Creates Fondazione Umberto Veronesi (FUV, Milan) awards from the
foundation's own register of fellowship holders, "I nostri ricercatori"
(menu: "Borse di ricerca"), https://www.fondazioneveronesi.it/ricerca-sui-tumori/i-nostri-ricercatori.
Every researcher the foundation funds through its annual calls (bandi) gets
a profile listing each funded project with its year, host institution and
duration. One award = one (researcher x funded project-year).
**1,499 fellowship-years, 2016-2026, from 929 researcher profiles (945 listed; 16 list no project)** (local run 2026-10-01).

**Prerequisites:**
- Run `scripts/local/fondazione_veronesi_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/fondazione_veronesi/fondazione_veronesi_projects.parquet`
  (runbook §1.4 shrink guard). Profiles are enumerated via the site's WP REST
  API (`back.fondazioneveronesi.it/wp-json/wp/v2/researchers`, allowed by
  that host's robots.txt) and read from the public profile pages
  (server-rendered Nuxt data).

**Source quirks handled in the script:**
- 33 project entries from 2016-2017 profiles were migrated with an interview
  block in the project fields (title = "Progetto" or a question such as "Se ti
  dico ricerca, cosa ti viene in mente?", host = the answer). The script
  recovers the real project title from the description and leaves host and
  description NULL for those rows (`qa_repaired`, `title_raw` in the raw table).
- 3 project-years listed twice in the same profile are kept once.

**Field notes:**
- `display_name`: the project title (Italian). `description`: the project
  summary. `funding_type`: `fellowship` (scope: fellowships are in scope;
  no filter applied). `funder_scheme`: 'Borsa di ricerca' (the profile does
  not say which call: post-doctoral, Pink is Good, etc.).
- Dates: each project names one funding year and a duration ("12 mesi" for
  ~98%, "24 mesi" otherwise). `start_date` = 1 January of that year (calls
  fund calendar-year fellowships; exact start dates are not published),
  `end_date` = start + duration - 1 day.
- `lead_investigator`: the fellow. Names are published "Family Given"; the
  script splits them using the profile slug ("given-family"), falling back to
  first-token-is-family with Italian particles. Affiliation = the published
  host ("dove", e.g. "ISM-CNR, Monterotondo (Roma)").
- `amount`/`currency`: NULL. Amounts are not published per fellowship.

**`funder_award_id` (§2.1.1):** citing works use mixed forms ("FUV 2017-1072",
"2018-1914", "Post-doctoral Fellowship 2021"); none appear on the site. The
ingest uses the synthetic, stable key `FUV-{profile post id}-{year}` and
collapses onto **0** existing citation stubs (checked in the last cell).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320322823` (exactly one row in `openalex.funders.funders`; no twin
  by name or ROR)
- display_name: Fondazione Umberto Veronesi
- ror_id / doi: from `openalex.funders.funders` (ror 041g7jm83, doi 10.13039/501100004710)

**Priority:** `587` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.fondazione_veronesi_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/fondazione_veronesi/fondazione_veronesi_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects, COUNT(DISTINCT funder_award_id) as distinct_ids,
       COUNT(DISTINCT researcher_post_id) as researchers
FROM openalex.awards.fondazione_veronesi_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.fondazione_veronesi_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.fondazione_veronesi_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320322823 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320322823;

## Step 2: Create Fondazione Veronesi Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.fondazione_veronesi_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320322823  -- Fondazione Umberto Veronesi
),

g AS (
    SELECT
        r.*,
        TRIM(r.funder_award_id) AS award_key,
        CASE WHEN TRY_CAST(r.first_year AS INT) IS NOT NULL THEN make_date(TRY_CAST(r.first_year AS INT), 1, 1) END AS start_dt,
        TRY_CAST(regexp_extract(r.duration, '([0-9]+)', 1) AS INT) AS months,
        CASE WHEN NULLIF(TRIM(r.lead_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(r.lead_given_name), ''),
            'family_name', NULLIF(TRIM(r.lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(r.`where`), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead
    FROM openalex.awards.fondazione_veronesi_raw r
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_key)))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    g.award_key as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'fellowship' as funding_type,
    'Borsa di ricerca' as funder_scheme,
    'fuv_ricercatori' as provenance,
    g.start_dt as start_date,
    CASE WHEN g.start_dt IS NOT NULL AND g.months IS NOT NULL
         THEN date_sub(add_months(g.start_dt, g.months), 1) END as end_date,
    YEAR(g.start_dt) as start_year,
    YEAR(CASE WHEN g.start_dt IS NOT NULL AND g.months IS NOT NULL
              THEN date_sub(add_months(g.start_dt, g.months), 1) END) as end_year,
    g.lead as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN g.lead IS NOT NULL THEN array(g.lead) END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_key)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f
WHERE g.award_key IS NOT NULL AND g.award_key != '';

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fuv_ricercatori' AND priority = 587;

-- Insert into openalex_awards_raw with priority.
-- Priority 587: direct-from-funder ingest of the Fondazione Umberto Veronesi
-- fellowship register. Higher wins under the 2026-06-20 DESC dedup.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    587 as priority
FROM openalex.awards.fondazione_veronesi_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.fondazione_veronesi_awards;

In [ ]:
%sql
-- §2.1.1 shape check: every funder_award_id is FUV-{post id}-{year}[-n].
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^FUV-[0-9]+-(19|20)[0-9]{2}(-[0-9]+)?$' THEN 1 ELSE 0 END) as ok_shape,
    SUM(CASE WHEN funder_award_id NOT RLIKE '^FUV-[0-9]+-(19|20)[0-9]{2}(-[0-9]+)?$' THEN 1 ELSE 0 END) as bad_shape
FROM openalex.awards.fondazione_veronesi_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.fondazione_veronesi_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, COUNT(DISTINCT lead_investigator.family_name) as fellows
FROM openalex.awards.fondazione_veronesi_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail
-- (a fellow renewed for up to ~5 consecutive years is expected).
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.fondazione_veronesi_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.fondazione_veronesi_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name AS host, COUNT(*) AS n
FROM openalex.awards.fondazione_veronesi_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 coverage (expect ~100% title/dates/PI, no amounts).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.fondazione_veronesi_awards;

In [ ]:
%sql
-- Overlap with existing citation stubs for this funder (expect 0: synthetic key).
SELECT c.provenance, COUNT(*) as stubs, COUNT(t.id) as collapsed_onto_fuv_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.fondazione_veronesi_awards t ON t.id = c.id
WHERE c.funder_id = 4320322823 AND c.provenance != 'fuv_ricercatori'
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fuv_ricercatori'
GROUP BY provenance, priority;